# TP-2: CleaningData - Applying OOP and Design Patterns

## Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

**Objective:**
- Combine all previous concepts into one cleaning pipeline.
- Use the **Template Method Pattern** to define a consistent workflow.

**Concept:** The Template Method defines the **skeleton** of a process and lets subclasses override specific steps.

**Instructions:**
1. Create an abstract class `DataPipeline` with a `run()` method defining the steps:
   `load()` → `clean()` → `transform()` → `save()`
2. Create `CSVDataPipeline` that implements each step using the previous classes.

In [1]:
import os
import time
import functools
from abc import ABC, abstractmethod

import pandas as pd

### Reused code from Exercises 1-4
| Exercise | Classes |
|---|---|
| 3 - Decorator | `log_action`, `log_time` |
| 1 - OOP | `CSVReader` (with decorators) |
| 2 - Strategy | `MissingValueStrategy`, `DropMissing`, `FillMean`, `FillMode`, `DataCleaner` |
| 4 - Factory | `DataTransform`, `NormalizeColumns`, `RemoveDuplicates`, `StandardizeText`, `TransformFactory` |

In [2]:
# Exercise 3: decorators

def log_action(func):
    """Decorator: log when a function/method starts and finishes."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] Starting '{func.__qualname__}'...")
        result = func(*args, **kwargs)
        print(f"[LOG] Finished '{func.__qualname__}'")
        return result
    return wrapper


def log_time(func):
    """Decorator: measure and print the execution time of a function/method."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[TIME] '{func.__qualname__}' took {elapsed:.4f} seconds")
        return result
    return wrapper

In [3]:
# Exercise 1: CSVReader (decorated in Exercise 3)

class CSVReader:
    """Encapsulates reading and previewing a CSV file."""

    def __init__(self, file_path):
        self.file_path = file_path
        self._data = None  # private: loaded DataFrame

    @log_action
    @log_time
    def read(self):
        """Load the CSV file into a pandas DataFrame and return it."""
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"File not found: {self.file_path}")
        self._data = pd.read_csv(self.file_path)
        return self._data

    @log_action
    def preview(self, n=5):
        """Print the top n rows of the data."""
        if self._data is None:
            self.read()
        print(f"Top {n} rows of '{self.file_path}':")
        print(self._data.head(n))

    @property
    def data(self):
        """Read-only access to the loaded DataFrame."""
        return self._data

    def __repr__(self):
        status = "loaded" if self._data is not None else "not loaded"
        return f"CSVReader(file_path='{self.file_path}', status={status})"

In [4]:
# Exercise 2: Strategy Pattern
class MissingValueStrategy(ABC):
    """Interface: every strategy must implement handle(df)."""

    @abstractmethod
    def handle(self, df):
        """Return a new DataFrame with missing values handled."""
        pass

class DropMissing(MissingValueStrategy):
    """Remove every row that has at least one missing value."""

    def handle(self, df):
        return df.dropna().reset_index(drop=True)


class FillMean(MissingValueStrategy):
    """Fill missing values in numeric columns with the column mean."""

    def handle(self, df):
        df = df.copy()
        num_cols = df.select_dtypes(include="number").columns
        df[num_cols] = df[num_cols].fillna(df[num_cols].mean())
        return df


class FillMode(MissingValueStrategy):
    """Fill missing values in every column with its most frequent value."""

    def handle(self, df):
        df = df.copy()
        for col in df.columns:
            if df[col].isnull().any():
                df[col] = df[col].fillna(df[col].mode()[0])
        return df


class DataCleaner:
    """Context class: applies whichever strategy it is given."""

    def __init__(self, strategy):
        self._strategy = strategy

    def set_strategy(self, strategy):
        """Switch to another strategy at runtime."""
        self._strategy = strategy

    @log_action
    @log_time
    def clean(self, df):
        print(f"Applying strategy: {type(self._strategy).__name__}")
        return self._strategy.handle(df)

In [5]:
# Exercise 4: Factory Pattern
class DataTransform(ABC):
    """Interface: every transformation must implement apply(df)."""

    @abstractmethod
    def apply(self, df):
        """Return a new, transformed DataFrame."""
        pass

class NormalizeColumns(DataTransform):
    """Min-max scale numeric columns to the range [0, 1]."""

    def __init__(self, columns=None):
        self.columns = columns  # None = all numeric columns

    def apply(self, df):
        df = df.copy()
        cols = self.columns or df.select_dtypes(include="number").columns.tolist()
        for col in cols:
            col_min, col_max = df[col].min(), df[col].max()
            if col_max == col_min:
                df[col] = 0.0
            else:
                df[col] = (df[col] - col_min) / (col_max - col_min)
        return df


class RemoveDuplicates(DataTransform):
    """Drop rows that are exact duplicates of a previous row."""

    def apply(self, df):
        before = len(df)
        df = df.drop_duplicates().reset_index(drop=True)
        print(f"Removed {before - len(df)} duplicate row(s)")
        return df


class StandardizeText(DataTransform):
    """Strip extra spaces and convert text columns to Title Case."""

    def apply(self, df):
        df = df.copy()
        text_cols = df.select_dtypes(exclude="number").columns
        for col in text_cols:
            df[col] = df[col].str.strip().str.title()
        return df


class TransformFactory:
    """Factory: creates a transformation object from a string name."""

    _transforms = {
        "normalize": NormalizeColumns,
        "remove_duplicates": RemoveDuplicates,
        "standardize_text": StandardizeText,
    }

    @classmethod
    def create(cls, name, **kwargs):
        key = name.strip().lower()
        if key not in cls._transforms:
            raise ValueError(f"Unknown transform '{name}'. Available: {cls.available()}")
        return cls._transforms[key](**kwargs)

    @classmethod
    def available(cls):
        return list(cls._transforms)

### Step 1: Abstract class `DataPipeline` (the template)
- `run()` is the **template method**: it fixes the **order** of the steps and is the same for every pipeline.
- `load()`, `clean()`, `transform()`, `save()` are **abstract steps**: each subclass decides *how* to do them.
- The DataFrame is passed from one step to the next.

In [6]:
class DataPipeline(ABC):
    """Template Method Pattern: run() defines the skeleton of the workflow."""

    @log_time
    def run(self):
        """Template method - the order of steps is fixed here."""
        print("========== Pipeline started ==========")
        df = self.load()
        df = self.clean(df)
        df = self.transform(df)
        self.save(df)
        print("========== Pipeline finished ==========")
        return df

    @abstractmethod
    def load(self):
        pass

    @abstractmethod
    def clean(self, df):
        pass

    @abstractmethod
    def transform(self, df):
        pass

    @abstractmethod
    def save(self, df):
        pass

### Step 2: `CSVDataPipeline` (implements every step with previous classes)
| Step | Uses |
|---|---|
| `load()` | `CSVReader` (Exercise 1) |
| `clean()` | `DataCleaner` + a strategy (Exercise 2) |
| `transform()` | `TransformFactory` (Exercise 4) |
| `save()` | `DataFrame.to_csv` |
| all steps | `@log_action` / `@log_time` (Exercise 3) |

`transforms` is a list of names, or `(name, options)` pairs when a transform needs arguments.

In [7]:
class CSVDataPipeline(DataPipeline):
    """Concrete pipeline: CSV file -> clean -> transform -> CSV file."""

    def __init__(self, input_path, output_path, strategy, transforms):
        self.input_path = input_path
        self.output_path = output_path
        self.cleaner = DataCleaner(strategy)  # composition
        self.transforms = transforms

    def load(self):
        return CSVReader(self.input_path).read()

    def clean(self, df):
        return self.cleaner.clean(df)

    @log_action
    def transform(self, df):
        for item in self.transforms:
            name, options = (item, {}) if isinstance(item, str) else item
            df = TransformFactory.create(name, **options).apply(df)
        return df

    @log_action
    def save(self, df):
        df.to_csv(self.output_path, index=False)
        print(f"Saved {len(df)} rows to '{self.output_path}'")

### Test: run the full pipeline on `sample_data.csv`

In [8]:
pipeline = CSVDataPipeline(
    input_path="sample_data.csv",
    output_path="cleaned_data.csv",
    strategy=FillMean(),
    transforms=[
        "standardize_text",
        "remove_duplicates",
        ("normalize", {"columns": ["age", "height_cm", "weight_kg", "score"]}),
    ],
)
result = pipeline.run()
result

========== Pipeline started ==========
[LOG] Starting 'CSVReader.read'...
[TIME] 'CSVReader.read' took 0.0063 seconds
[LOG] Finished 'CSVReader.read'
[LOG] Starting 'DataCleaner.clean'...
Applying strategy: FillMean
[TIME] 'DataCleaner.clean' took 0.0048 seconds
[LOG] Finished 'DataCleaner.clean'
[LOG] Starting 'CSVDataPipeline.transform'...
Removed 0 duplicate row(s)
[LOG] Finished 'CSVDataPipeline.transform'
[LOG] Starting 'CSVDataPipeline.save'...
Saved 7 rows to 'cleaned_data.csv'
[LOG] Finished 'CSVDataPipeline.save'
========== Pipeline finished ==========
[TIME] 'DataPipeline.run' took 0.0192 seconds


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,0.25000,0.333333,0.433333,New York,0.588235
1,2,Bob,0.40625,0.800000,0.500000,Los Angeles,0.882353
2,3,Charlie,1.00000,0.533333,0.566667,Chicago,0.647059
3,4,David,0.40625,0.466667,0.833333,Houston,0.000000
4,5,Eva,0.00000,0.000000,0.000000,New York,0.764706
5,6,Frank,0.37500,1.000000,1.000000,Los Angeles,0.647059
6,7,Grace,0.40625,0.133333,0.166667,Chicago,1.000000


In [9]:
# Check the saved file
saved = CSVReader("cleaned_data.csv").read()
print("Missing values in saved file:", saved.isnull().sum().sum())
saved

[LOG] Starting 'CSVReader.read'...
[TIME] 'CSVReader.read' took 0.0021 seconds
[LOG] Finished 'CSVReader.read'
Missing values in saved file: 0


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,0.25000,0.333333,0.433333,New York,0.588235
1,2,Bob,0.40625,0.800000,0.500000,Los Angeles,0.882353
2,3,Charlie,1.00000,0.533333,0.566667,Chicago,0.647059
3,4,David,0.40625,0.466667,0.833333,Houston,0.000000
4,5,Eva,0.00000,0.000000,0.000000,New York,0.764706
5,6,Frank,0.37500,1.000000,1.000000,Los Angeles,0.647059
6,7,Grace,0.40625,0.133333,0.166667,Chicago,1.000000


### Override one step in a subclass
The Template Method lets a subclass change **only one step** while `run()` stays the same.
`PreviewPipeline` overrides `save()` to print the result instead of writing a file, and we also switch the strategy to `DropMissing`.

In [10]:
class PreviewPipeline(CSVDataPipeline):
    """Same workflow, but save() only previews the result."""

    def save(self, df):
        print(f"Preview only - nothing saved ({len(df)} rows):")
        print(df)


preview = PreviewPipeline(
    input_path="sample_data.csv",
    output_path=None,
    strategy=DropMissing(),
    transforms=["standardize_text"],
)
preview.run();

========== Pipeline started ==========
[LOG] Starting 'CSVReader.read'...
[TIME] 'CSVReader.read' took 0.0010 seconds
[LOG] Finished 'CSVReader.read'
[LOG] Starting 'DataCleaner.clean'...
Applying strategy: DropMissing
[TIME] 'DataCleaner.clean' took 0.0033 seconds
[LOG] Finished 'DataCleaner.clean'
[LOG] Starting 'CSVDataPipeline.transform'...
[LOG] Finished 'CSVDataPipeline.transform'
Preview only - nothing saved (2 rows):
   id   name   age  height_cm  weight_kg      city  score
0   1  Alice  29.0      165.0       68.0  New York   85.0
1   5    Eva  27.0      160.0       55.0  New York   88.0
========== Pipeline finished ==========
[TIME] 'DataPipeline.run' took 0.0062 seconds


### Summary
- **Template Method**: `DataPipeline.run()` fixes the workflow `load → clean → transform → save`; subclasses implement or override single steps.
- **OOP (Ex. 1)**: `CSVReader` loads the data.
- **Strategy (Ex. 2)**: the missing-value method is chosen by passing `FillMean()`, `DropMissing()`, ...
- **Decorator (Ex. 3)**: `@log_action` and `@log_time` show each step and its duration.
- **Factory (Ex. 4)**: transformations are created from strings like `"normalize"`.

Result: a complete, reusable and flexible CSV cleaning pipeline. Changing the strategy or the transformations only needs a different argument - no class has to be modified.